# YouTube Audio Scraper - Syllable Dataset Generator

## 📋 Project Overview
Pipeline untuk scraping audio dari YouTube, pemrosesan audio, transkripsi, dan ekstraksi fitur suku kata.

### 📁 Module Structure
- **config.py**: Konfigurasi global (sampling rate, model parameters, output paths)
- **audio_processor.py**: Semua fungsi utility untuk audio processing, transcription, dan feature extraction
- **scraper.py**: Main pipeline yang mengintegrasikan semua proses

### 🔄 Processing Pipeline
1. **Download Audio** dari YouTube
2. **Clean Audio** (denoising, bandpass filter 80-3000 Hz, normalisasi volume)
3. **Transcribe** menggunakan WhisperX
4. **Syllabify** kata menjadi suku kata
5. **Segment Audio** per suku kata
6. **Extract Features** (MFCC, RMS, ZCR, F0, spectral centroid, dll)
7. **Save Dataset** ke CSV

### 📊 Output Features
- Duration, ZCR, RMS, Spectral Centroid
- 13 MFCC Coefficients (mean & std)
- Fundamental Frequency (F0)


## 🔧 Setup & Dependencies
Mengimport semua library yang diperlukan dan inisialisasi konfigurasi global.

**Output Penting:**
- ✓ Konfirmasi semua module berhasil diimport
- 📁 Base directory untuk menyimpan dataset
- ⚙️ Konfigurasi: sampling rate, model Whisper
- 📊 Info tentang run sebelumnya (jika ada)
- 📈 Index untuk run berikutnya

In [1]:
# Import Modules
import os
import sys
from pathlib import Path

# Standard libraries
import numpy as np
import pandas as pd
import librosa
import noisereduce as nr
from pydub import AudioSegment

# Add current directory to path untuk import modules lokal
current_dir = Path.cwd()
if str(current_dir) not in sys.path:
    sys.path.insert(0, str(current_dir))

# Import custom modules dari libs folder
from libs import config, scraper, path_manager
from libs.audio_processor import validate_youtube_url
from libs.path_manager import setup_run_directories, get_run_info

print("✓ Semua module berhasil diimport dari libs/")
print(f"✓ Base directory: {config.BASE_SCRAPED_DIR}")
print(f"✓ Config: SAMPLE_RATE={config.SAMPLE_RATE}, WHISPER_MODEL={config.WHISPER_MODEL}")

# Check existing runs
run_info = get_run_info(config.BASE_SCRAPED_DIR)
print(f"✓ Total runs sebelumnya: {run_info['total_runs']}")
if run_info['runs']:
    print(f"  Run IDs: {run_info['runs']}")
    print(f"✓ Next run akan menggunakan index: {run_info['next_index']}")

c:\Users\ASUS\Documents\CC x DBS 2026 (DS)\Capstone Project\Heartz\data-science2\scraping\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Semua module berhasil diimport dari libs/
✓ Base directory: scraped
✓ Config: SAMPLE_RATE=16000, WHISPER_MODEL=small
✓ Total runs sebelumnya: 0


## ✅ System Requirements Check
Verifikasi FFmpeg installation yang diperlukan untuk download audio dari YouTube.

**Fungsi:**
- ✓ Cek apakah FFmpeg sudah terinstall
- 📋 Tampilkan lokasi FFmpeg executable
- 📝 Instruksi instalasi jika belum ada

In [2]:
# Check FFmpeg Installation
print("🔍 Checking system requirements...\n")

from libs import ffmpeg_checker

ffmpeg_status = ffmpeg_checker.print_ffmpeg_status()

if not ffmpeg_status:
    print("\n💡 NEXT STEPS:")
    print("   1. Install FFmpeg dari link di atas")
    print("   2. Restart terminal/notebook kernel")
    print("   3. Run this cell again\n")
    print("   NOTE: Anda masih bisa run pipeline tanpa FFmpeg,")
    print("   tapi dengan fallback method (lebih lambat)")


🔍 Checking system requirements...


🔍 FFMPEG STATUS CHECK
✅ FFmpeg:  Found at ffmpeg
✅ FFprobe: Found at ffprobe

💾 Local bundled FFmpeg: Not present



## 🔊 Extract & Enhance Audio Syllables
Mengekstraksi file audio individual per suku kata dengan enhancement (time stretching, denoising, normalisasi).

**⚠️ PENTING - Indexing Terpisah:**
- **Input**: Membaca dari `syllable_dataset_[N].csv` dan `temp_audio_[N]/` (scraper run index)
- **Output**: Menyimpan ke `extracted_syllables_[X]/`

**Konfigurasi:**
- `TARGET_SYLLABLES`: Set suku kata yang ingin diekstraksi (a, i, u, e, o, ma, mi, dll)
- `TARGET_MIN_DURATION`: Durasi minimum audio untuk enhancement (default 0.3 detik)

**Enhancement Techniques:**
1. **Time Stretching**: Memperpanjang audio yang terlalu pendek (faktor max 2.5x)
2. **Denoising**: Mengurangi background noise dengan noise reduction
3. **RMS Normalization**: Standarisasi volume (target RMS = 0.1)
4. **Fade In/Out**: Smooth transition di awal/akhir audio untuk menghindari clicks

**Fungsi:**
- `enhance_audio_chunk()`: Enhance individual audio chunk
- `extract_syllables_from_run(run_index)`: Extract dari scraper run spesifik atau latest run
- `get_next_extracted_syllables_index()`: Hitung next index untuk output (independen)

**Output:**
- Direktori terstruktur: `extracted_syllables_[X]/[label]/[filename].wav`
- Index [X] adalah **INDEPENDEN** dari scraper run index
- Setiap suku kata memiliki folder tersendiri
- Audio sudah enhancement-ready untuk augmentation

In [3]:
# Extract & Enhance Audio Syllables
# Gunakan cell ini untuk extract individual syllable audio files dari dataset yang sudah dibuat
# dengan enhancement (time stretching, normalisasi, denoising)

# ========== KONFIGURASI EXTRACTION ==========
TARGET_SYLLABLES = {
    'a', 'i', 'u', 'e', 'o',
    'ma', 'mi', 'mu', 'me', 'mo',
    'ba', 'bi', 'bu', 'be', 'bo',
    'pa', 'pi', 'pu', 'pe', 'po'
}
TARGET_MIN_DURATION = 0.3  # Durasi minimum untuk enhancement (detik)

# ========== HELPER FUNCTION: Get Next Index untuk Extracted Syllables ==========
def get_next_extracted_syllables_index(base_dir: str = config.BASE_SCRAPED_DIR) -> int:
    """
    Hitung next index untuk extracted_syllables folder (INDEPENDENT dari scraper run index)
    
    Returns:
        Index berikutnya untuk extracted_syllables_[X]
        
    Example:
        Jika ada extracted_syllables_1, extracted_syllables_2 → return 3
    """
    import re
    
    if not os.path.exists(base_dir):
        return 1
    
    indexes = []
    for item in os.listdir(base_dir):
        # Match extracted_syllables_[number]
        match = re.match(r"^extracted_syllables_(\d+)$", item)
        if match:
            try:
                idx = int(match.group(1))
                indexes.append(idx)
            except (ValueError, IndexError):
                continue
    
    return max(indexes) + 1 if indexes else 1

# ========== FUNGSI ENHANCEMENT AUDIO ==========
def enhance_audio_chunk(chunk_audio, target_duration=TARGET_MIN_DURATION, sr=config.SAMPLE_RATE):
    """
    Enhance audio chunk dengan time stretching, denoising, normalisasi
    """
    samples = np.array(chunk_audio.get_array_of_samples(), dtype=np.float32)
    if chunk_audio.channels == 2:
        samples = samples.reshape((-1, 2)).mean(axis=1)
    samples = samples / (2**15)
    
    if len(samples) == 0:
        return chunk_audio
    
    durasi = len(samples) / sr
    
    # 1. Time stretching jika terlalu pendek
    if durasi < target_duration:
        stretch_factor = min(target_duration / durasi, 2.5)
        samples = librosa.effects.time_stretch(samples, rate=1/stretch_factor)
    
    # 2. Denoising
    if len(samples) > sr * 0.2:
        samples = nr.reduce_noise(y=samples, sr=sr, stationary=True, prop_decrease=0.7)
    
    # 3. Normalisasi RMS
    rms = np.sqrt(np.mean(samples**2))
    if rms > 0:
        samples = samples * (0.1 / rms)
        samples = np.clip(samples, -0.95, 0.95)
    
    # 4. Fade in/out
    fade_len = int(0.01 * sr)
    if len(samples) > fade_len * 2:
        samples[:fade_len] *= np.linspace(0, 1, fade_len)
        samples[-fade_len:] *= np.linspace(1, 0, fade_len)
    
    samples_int16 = (samples * 32767).astype(np.int16)
    return AudioSegment(samples_int16.tobytes(), frame_rate=sr, sample_width=2, channels=1)


# ========== MAIN EXTRACTION FUNCTION ==========
def extract_syllables_from_run(run_index: int = None):
    """
    Extract audio files dari specific run dengan enhancement.
    
    ⚠️ IMPORTANT: Output folder menggunakan INDEX TERPISAH untuk extracted_syllables
    - run_index: Index dari scraper run (untuk membaca CSV dan audio file)
    - extracted_index: Index independen untuk output folder extracted_syllables_[X]
    """
    if run_index is None:
        # Get latest run
        run_info = get_run_info(config.BASE_SCRAPED_DIR)
        if run_info['total_runs'] == 0:
            print("❌ Belum ada dataset")
            return
        run_index = max(run_info['runs'])
    
    # Setup paths untuk INPUT (membaca dari scraper run)
    csv_file = os.path.join(config.BASE_SCRAPED_DIR, f"syllable_dataset_{run_index}.csv")
    temp_audio_dir = os.path.join(config.BASE_SCRAPED_DIR, f"temp_audio_{run_index}")
    
    # Setup OUTPUT dengan INDEX TERPISAH untuk extracted_syllables
    extracted_index = get_next_extracted_syllables_index()
    output_dir = os.path.join(config.BASE_SCRAPED_DIR, f"extracted_syllables_{extracted_index}")
    
    # Check files exist
    if not os.path.exists(csv_file):
        print(f"❌ Dataset tidak ditemukan: {csv_file}")
        return
    
    # Find audio file in temp_audio_dir
    audio_path = None
    if os.path.exists(temp_audio_dir):
        for f in os.listdir(temp_audio_dir):
            if f.endswith("_cleaned.wav"):
                audio_path = os.path.join(temp_audio_dir, f)
                break
    
    if not audio_path:
        print(f"❌ Audio file tidak ditemukan di: {temp_audio_dir}")
        return
    
    print(f"📁 Extraction dari Run #{run_index}")
    print(f"   CSV: {csv_file}")
    print(f"   Audio: {audio_path}")
    
    # Read CSV dan filter
    df = pd.read_csv(csv_file)
    df_target = df[df['syllable'].isin(TARGET_SYLLABLES)]
    
    if df_target.empty:
        print(f"❌ Tidak ada target syllables di dataset")
        return
    
    # Extract
    audio = AudioSegment.from_wav(audio_path)
    os.makedirs(output_dir, exist_ok=True)
    
    exported = 0
    for idx, row in df_target.iterrows():
        try:
            syll = row['syllable']
            start_ms = int(row['syllable_start_sec'] * 1000)
            end_ms = int(row['syllable_end_sec'] * 1000)
            
            if start_ms < 0 or end_ms > len(audio):
                continue
            
            chunk = audio[start_ms:end_ms]
            chunk_enhanced = enhance_audio_chunk(chunk)
            
            filename = f"{row['original_word']:.3f}_{start_ms:.0f}ms.wav"
            syll_dir = os.path.join(output_dir, syll)
            os.makedirs(syll_dir, exist_ok=True)
            
            chunk_enhanced.export(os.path.join(syll_dir, filename), format="wav")
            exported += 1
        except Exception as e:
            continue
    
    print(f"✅ Extracted {exported} files ke: {output_dir}")


# Run extraction
print("💡 Gunakan extract_syllables_from_run() untuk extract audio files")
print("   Contoh: extract_syllables_from_run(1)  # Extract dari run 1")
print("   Atau: extract_syllables_from_run()     # Extract dari latest run")

💡 Gunakan extract_syllables_from_run() untuk extract audio files
   Contoh: extract_syllables_from_run(1)  # Extract dari run 1
   Atau: extract_syllables_from_run()     # Extract dari latest run


## 🚀 Usage Examples & Single Video Processing
Memproses single YouTube video dengan pipeline lengkap scraping & feature extraction.

**Cara Kerja:**
1. Validasi URL YouTube untuk memastikan valid dan accessible
2. Download audio dari YouTube
3. Transcribe menggunakan WhisperX
4. Syllabify dan segment audio
5. Extract features audio per suku kata
6. Save ke CSV dengan indexed naming

**Catatan Penting:**
- ⚠️ **Gunakan video PUBLIC** yang tidak dihapus/private
- 🔒 Video yang di-private atau sudah dihapus akan error
- 📁 Setiap run akan otomatis generate folder baru dengan index: `temp_audio_1`, `temp_audio_2`, dst
- 📊 CSV output: `syllable_dataset_[N].csv`

**Output Yang Dihasilkan:**
- ✅ Confirmasi: URL valid
- 📹 Link video berhasil diproses
- 📈 Jumlah syllables yang berhasil diekstraksi
- 📋 Preview data: 5 baris pertama dataset

### Step 1: Validate YouTube URL
Validasi URL YouTube sebelum processing untuk menghindari error.

**Tips:**
- ✓ Gunakan link yang bersifat public dan tidak likely akan dihapus
- ✓ Rick Roll video (contoh) adalah salah satu video yang stabil dan selalu available
- ℹ️ Ganti `youtube_url` dengan URL Anda sendiri jika ingin test
- 💾 Link akan digunakan untuk `youtube_url_input` di cell berikutnya

In [4]:
# Example 1: Process Single YouTube Video
# 🎯 PENTING: Gunakan video YouTube yang PUBLIC dan tidak dihapus
# Contoh video yang PASTI berfungsi:

youtube_url = "https://youtu.be/MVhL6wPhrPs?si=bZPKJPt94EvgKZaB"

# Validate URL
if validate_youtube_url(youtube_url):
    print(f"✓ URL valid: {youtube_url}")
else:
    print(f"✗ URL tidak valid: {youtube_url}")

# Atau gunakan video lain yang public:
# - https://www.youtube.com/watch?v=9bZkp7q19f0  # PSY - Gangnam Style
# - https://www.youtube.com/watch?v=kffacxfA7g4  # David Guetta - Titanium

✓ URL valid: https://youtu.be/MVhL6wPhrPs?si=bZPKJPt94EvgKZaB


### Step 2: Process YouTube Video with Pipeline
Menjalankan full pipeline scraping dengan automatic indexed paths.

**Proses:**
1. Prompt user untuk input YouTube URL (atau tekan Enter untuk skip)
2. Validate URL untuk memastikan accessible
3. Download audio & process dengan full pipeline
4. Simpan hasil ke dataset CSV dengan index otomatis

**Output Sukses:**
- ✅ Konfirmasi dataset berhasil dibuat
- 📊 Run Index yang dihasilkan (untuk reference di cell lain)
- 📁 Paths: temp audio dir, dataset file location
- 📈 Shape (jumlah rows/columns)
- 👁️ Preview 5 baris pertama dari dataset

**Troubleshooting:**
- ❌ Jika gagal, cek apakah URL valid dan video masih public
- 🔧 Pastikan FFmpeg sudah terinstall
- 💾 Hasil disimpan dengan automatic indexing, jadi tidak akan overwrite

In [6]:
# Run Pipeline dengan Indexed Paths
# Setiap kali Anda menjalankan cell ini dengan YouTube URL baru,
# folder dan CSV akan otomatis diberi index (temp_audio_1, temp_audio_2, dll)

youtube_url_input = input("Masukkan YouTube URL (atau tekan Enter untuk skip): ").strip()

if youtube_url_input:
    print(f"\n📹 Processing: {youtube_url_input}")
    
    # Process video dengan indexed paths
    df_result, run_info = scraper.process_youtube_video_indexed(youtube_url_input)
    
    if df_result is not None:
        print(f"\n✅ Dataset berhasil dibuat!")
        print(f"   Run Index: {run_info['run_index']}")
        print(f"   Temp Audio Dir: {run_info['temp_audio_dir']}")
        print(f"   Dataset File: {run_info['dataset_filename']}")
        print(f"   Shape: {df_result.shape}")
        print(f"\nFirst few rows:")
        print(df_result.head())
    else:
        print("❌ Processing gagal")
else:
    print("Skipped. Masukkan YouTube URL untuk memproses")


📹 Processing: https://youtu.be/MVhL6wPhrPs?si=bZPKJPt94EvgKZaB

RUN #1 - YOUTUBE VIDEO SCRAPER
📁 Temp Audio: scraped\temp_audio_1
📊 Output Dataset: scraped\syllable_dataset_1.csv

[1/6] Mengunduh audio dari YouTube...
   - Downloading dengan FFmpeg + yt-dlp CLI (Node.js runtime)...
   ✓ Download OK (158.9MB): original_audio.wav
   ✓ Audio asli: scraped\temp_audio_1\original_audio.wav

[2/6] Membersihkan audio...
   - Denoising spectral gating
   - Bandpass filter (80-3000 Hz)
   - Normalisasi RMS
   ✓ Audio bersih disimpan ke: scraped\temp_audio_1\original_audio_cleaned.wav
   ✓ Audio bersih: scraped\temp_audio_1\original_audio_cleaned.wav

[3/6] Transkripsi audio dengan WhisperX...
   - Menggunakan device: cpu
   - Loading WhisperX model 'small'...
   - Language: Bahasa Indonesia (id)


c:\Users\ASUS\Documents\CC x DBS 2026 (DS)\Capstone Project\Heartz\data-science2\scraping\venv\Lib\site-packages\pyannote\audio\core\io.py:47: UserWarning: 
torchcodec is not installed correctly so built-in audio decoding will fail. Solutions are:
* use audio preloaded in-memory as a {'waveform': (channel, time) torch.Tensor, 'sample_rate': int} dictionary;
* fix torchcodec installation. Error message was:

Could not load libtorchcodec. Likely causes:
          1. FFmpeg is not properly installed in your environment. We support
             versions 4, 5, 6 and 7.
          2. The PyTorch version (2.8.0+cpu) is not compatible with
             this version of TorchCodec. Refer to the version compatibility
             table:
             https://github.com/pytorch/torchcodec?tab=readme-ov-file#installing-torchcodec.
          3. Another runtime dependency; see exceptions below.
        The following exceptions were raised as we tried to load libtorchcodec:
        
[start of libtorchco

2026-05-20 12:34:43 - whisperx.asr - INFO - No language specified, language will be detected for each audio file (increases inference time)
2026-05-20 12:34:43 - whisperx.vads.pyannote - INFO - Performing voice activity detection using Pyannote...


Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.1. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint c:\Users\ASUS\Documents\CC x DBS 2026 (DS)\Capstone Project\Heartz\data-science2\scraping\venv\Lib\site-packages\whisperx\assets\pytorch_model.bin`


   - Loading alignment model...


...

   - Kata 468: 'Sembilan' → ['se', 'mbi', 'lan']
   - Kata 469: 'Sembilan' → ['se', 'mbi', 'lan']
   - Kata 470: 'Sembilan' → ['se', 'mbi', 'lan']
   - Kata 471: 'Wah,' → ['wah,']
   - Kata 472: 'tinggi' → ['ti', 'nggi']
   - Kata 473: 'banget' → ['ba', 'nget']
   - Kata 474: 'ya!' → ['ya!']
   - Kata 475: 'Ada' → ['a', 'da']
   - Kata 476: 'berapa' → ['be', 'ra', 'pa']


c:\Users\ASUS\Documents\CC x DBS 2026 (DS)\Capstone Project\Heartz\data-science2\scraping\venv\Lib\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1488
  warnings.warn(
c:\Users\ASUS\Documents\CC x DBS 2026 (DS)\Capstone Project\Heartz\data-science2\scraping\venv\Lib\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1504
  warnings.warn(


   - Kata 477: 'baluk' → ['ba', 'luk']
   - Kata 478: 'ya?' → ['ya?']
   - Kata 479: 'Kita' → ['ki', 'ta']
   - Kata 480: 'hitung' → ['hi', 'tung']
   - Kata 481: 'sama-sama,' → ['sa', 'ma', '-sa', 'ma,']
   - Kata 482: 'yuk!' → ['yuk!']
   - Kata 483: 'Satu' → ['sa', 'tu']
   - Kata 484: 'Dua' → ['du', 'a']
   - Kata 485: 'Tiga,' → ['ti', 'ga,']
   - Kata 486: 'empat,' → ['e', 'mpat,']
   - Kata 487: 'lima,' → ['li', 'ma,']
   - Kata 488: 'enam,' → ['e', 'nam,']
   - Kata 489: 'tujuh,' → ['tu', 'juh,']
   - Kata 490: 'delapan,' → ['de', 'la', 'pan,']
   - Kata 491: 'sembilan,' → ['se', 'mbi', 'lan,']
   - Kata 492: 'sepuluh.' → ['se', 'pu', 'luh.']
   - Kata 493: 'Ada' → ['a', 'da']
   - Kata 494: 'sepuluh.' → ['se', 'pu', 'luh.']
   - Kata 495: 'Sepuluh.' → ['se', 'pu', 'luh.']
   - Kata 496: 'ada' → ['a', 'da']
   - Kata 497: 'sepuluh' → ['se', 'pu', 'luh']
   - Kata 498: 'sepuluh' → ['se', 'pu', 'luh']
   - Kata 499: 'ada' → ['a', 'da']
   - Kata 500: 'berapa?' → ['be', 'ra', 'pa?'

c:\Users\ASUS\Documents\CC x DBS 2026 (DS)\Capstone Project\Heartz\data-science2\scraping\venv\Lib\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1776
  warnings.warn(


   - Kata 512: 'kerasa' → ['ke', 'ra', 'sa']
   - Kata 513: 'Kita' → ['ki', 'ta']
   - Kata 514: 'udah' → ['u', 'dah']
   - Kata 515: 'hitung' → ['hi', 'tung']
   - Kata 516: 'sampai' → ['sa', 'mpa', 'i']
   - Kata 517: 'sepuluh.' → ['se', 'pu', 'luh.']
   - Kata 518: 'Nah,' → ['nah,']
   - Kata 519: 'ke' → ['ke']
   - Kata 520: 'rasanya.' → ['ra', 'sa', 'nya.']
   - Kata 521: 'Nontonnya' → ['no', 'nto', 'nnya']
   - Kata 522: 'jangan' → ['ja', 'ngan']
   - Kata 523: 'lama-lama.' → ['la', 'ma', '-la', 'ma.']
   - Kata 524: 'Kita' → ['ki', 'ta']
   - Kata 525: 'main' → ['ma', 'in']
   - Kata 526: 'mainan' → ['ma', 'i', 'nan']
   - Kata 527: 'aja' → ['a', 'ja']
   - Kata 528: 'sambil' → ['sa', 'mbil']
   - Kata 529: 'belajar.' → ['be', 'la', 'jar.']
   - Kata 530: 'Gimana?' → ['gi', 'ma', 'na?']
   - Kata 531: 'Seru' → ['se', 'ru']
   - Kata 532: 'kan?' → ['kan?']
   - Kata 533: 'Tapi' → ['ta', 'pi']
   - Kata 534: 'jangan' → ['ja', 'ngan']
   - Kata 535: 'lupa' → ['lu', 'pa']
   - Kata 

## 📊 Dataset Analysis & Inspection
Memuat dan menganalisis dataset yang sudah di-generate dari processing.

**Fitur:**
- 📋 List semua available runs (jika ada lebih dari 1)
- 📈 Load dataset terbaru secara otomatis
- 🔍 Inspeksi struktur: columns, data types, shape
- 📊 Statistical summary (mean, std, min, max, dll)

**Output:**
- ✓ Dataset path yang diload
- 📏 Shape: (num_rows, num_columns)
- 📝 Column names (label, syllable, duration, MFCC_[0-12], F0, ZCR, RMS, dll)
- 📋 Data types per column
- 📊 Statistics: count, mean, std, 25%, 50%, 75%, max

**Catatan:**
- Cell ini REQUIRE dataset sudah di-generate sebelumnya
- Jika error "Belum ada dataset", jalankan processing cell terlebih dahulu

In [7]:
# Load dan Analyze Dataset dari Indexed Paths (ALL INDICES)
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Get run info
run_info = get_run_info(config.BASE_SCRAPED_DIR)

if run_info['total_runs'] == 0:
    print("❌ Belum ada dataset. Jalankan cell sebelumnya terlebih dahulu.")
else:
    all_runs = sorted(run_info['runs'])
    print(f"📊 Available runs: {all_runs}")
    print(f"📈 Total datasets: {len(all_runs)}\n")
    
    # ========== LOAD ALL DATASETS ==========
    all_datasets = {}
    
    for run_id in all_runs:
        csv_file = os.path.join(config.BASE_SCRAPED_DIR, f"syllable_dataset_{run_id}.csv")
        
        try:
            df = pd.read_csv(csv_file)
            all_datasets[run_id] = df
            print(f"✓ Run #{run_id}: {df.shape[0]} rows × {df.shape[1]} columns")
        except FileNotFoundError:
            print(f"✗ Run #{run_id}: File tidak ditemukan ({csv_file})")
    
    # ========== DETAILED INFO PER DATASET ==========
    print(f"\n{'='*70}")
    print("📋 DATASET ANALYSIS - PER INDEX")
    print(f"{'='*70}")
    
    for run_id in all_runs:
        if run_id in all_datasets:
            df = all_datasets[run_id]
            
            print(f"\n📁 RUN #{run_id}")
            print(f"{'─'*70}")
            
            # Column names
            print(f"\n📝 Column Names:")
            print(f"{df.columns.tolist()}")
            
            # Data types
            print(f"\n📊 Data Types:")
            print(df.dtypes)
            
            # Statistical summary
            print(f"\n📈 Statistical Summary:")
            print(df.describe())
    
    # ========== STORE DATASETS ==========
    print(f"\n{'='*70}")
    print("💾 Datasets Summary")
    print(f"{'='*70}")
    print(f"\n✓ Total datasets loaded: {len(all_datasets)}")
    print(f"✓ Access via: all_datasets[run_id]")
    print(f"\nAvailable indices: {list(all_datasets.keys())}")

📊 Available runs: [1]
📈 Total datasets: 1

✓ Run #1: 1149 rows × 35 columns

📋 DATASET ANALYSIS - PER INDEX

📁 RUN #1
──────────────────────────────────────────────────────────────────────

📝 Column Names:
['original_word', 'syllable', 'syllable_start_sec', 'syllable_end_sec', 'duration_sec', 'zcr', 'rms', 'spectral_centroid', 'f0_mean', 'mfcc_1_mean', 'mfcc_1_std', 'mfcc_2_mean', 'mfcc_2_std', 'mfcc_3_mean', 'mfcc_3_std', 'mfcc_4_mean', 'mfcc_4_std', 'mfcc_5_mean', 'mfcc_5_std', 'mfcc_6_mean', 'mfcc_6_std', 'mfcc_7_mean', 'mfcc_7_std', 'mfcc_8_mean', 'mfcc_8_std', 'mfcc_9_mean', 'mfcc_9_std', 'mfcc_10_mean', 'mfcc_10_std', 'mfcc_11_mean', 'mfcc_11_std', 'mfcc_12_mean', 'mfcc_12_std', 'mfcc_13_mean', 'mfcc_13_std']

📊 Data Types:
original_word             str
syllable                  str
syllable_start_sec    float64
syllable_end_sec      float64
duration_sec          float64
zcr                   float64
rms                   float64
spectral_centroid     float64
f0_mean             

## 🎵 Extract Audio Syllables untuk Augmentation
Ekstraksi potongan audio individual untuk setiap target syllable dengan enhancement otomatis.

**Tujuan:**
- ✅ Ekstrak audio per suku kata dari dataset yang sudah dibuat
- ✅ Simpan ke folder `extracted_syllables_[N]` dengan index sesuai scraper run
- ✅ Enhancement otomatis: time stretching, denoising, normalisasi volume
- ✅ Persiapan data untuk augmentation pipeline di augmentation.ipynb

**Indexing:**
- **Input**: Membaca dari scraper run #[N] (syllable_dataset_[N].csv, temp_audio_[N]/)
- **Output**: Menyimpan ke `extracted_syllables_[N]` (INDEX **SAMA DENGAN SCRAPER RUN**)
- **Overwrite**: Jika folder sudah ada, akan ditimpa dengan data terbaru
- Contoh: Extract dari scraper run #2 → output ke extracted_syllables_2/ (jika sudah ada, OVERWRITE)

**Target Syllables:**
- Vokal: a, i, u, e, o
- Consonant + Vokal: ma, mi, mu, me, mo, ba, bi, bu, be, bo, pa, pi, pu, pe, po
- Dapat dikustomisasi di code cell

**Enhancement Techniques:**
1. **Time Stretching**: Panjangkan audio pendek (max 2.5x)
2. **Denoising**: Kurangi background noise
3. **RMS Normalization**: Standarisasi volume (target 0.1)
4. **Fade In/Out**: Smooth transition di awal/akhir



In [8]:
# ========== KONFIGURASI EXTRACTION ==========
# TARGET SYLLABLES yang ingin diekstraksi
TARGET_SYLLABLES_EXTRACTION = {
    'a', 'i', 'u', 'e', 'o',
    'ma', 'mi', 'mu', 'me', 'mo',
    'ba', 'bi', 'bu', 'be', 'bo',
    'pa', 'pi', 'pu', 'pe', 'po'
}
TARGET_MIN_DURATION_FOR_ENHANCEMENT = 0.3  # Durasi minimum (detik)

print("✓ Extraction config loaded")
print(f"✓ Target syllables: {len(TARGET_SYLLABLES_EXTRACTION)} labels")
print(f"✓ Target min duration: {TARGET_MIN_DURATION_FOR_ENHANCEMENT}s")


# ========== HELPER: Enhance Audio Chunk ==========
def enhance_audio_chunk(chunk_audio, target_duration=TARGET_MIN_DURATION_FOR_ENHANCEMENT, sr=config.SAMPLE_RATE):
    """
    Enhance audio chunk dengan time stretching, denoising, normalisasi
    
    Steps:
    1. Time stretching jika durasi terlalu pendek (max 2.5x)
    2. Denoising untuk kurangi background noise
    3. RMS Normalization untuk standarisasi volume
    4. Fade in/out untuk smooth transition
    """
    import numpy as np
    from pydub import AudioSegment
    import librosa
    import noisereduce as nr
    
    try:
        samples = np.array(chunk_audio.get_array_of_samples(), dtype=np.float32)
        if chunk_audio.channels == 2:
            samples = samples.reshape((-1, 2)).mean(axis=1)
        samples = samples / (2**15)
        
        if len(samples) == 0:
            return chunk_audio
        
        durasi = len(samples) / sr
        
        # 1. Time stretching jika terlalu pendek
        if durasi < target_duration:
            stretch_factor = min(target_duration / durasi, 2.5)
            samples = librosa.effects.time_stretch(samples, rate=1/stretch_factor)
        
        # 2. Denoising (hanya jika audio cukup panjang)
        if len(samples) > sr * 0.2:
            samples = nr.reduce_noise(y=samples, sr=sr, stationary=True, prop_decrease=0.7)
        
        # 3. Normalisasi RMS
        rms = np.sqrt(np.mean(samples**2))
        if rms > 0:
            samples = samples * (0.1 / rms)
            samples = np.clip(samples, -0.95, 0.95)
        
        # 4. Fade in/out
        fade_len = int(0.01 * sr)
        if len(samples) > fade_len * 2:
            samples[:fade_len] *= np.linspace(0, 1, fade_len)
            samples[-fade_len:] *= np.linspace(1, 0, fade_len)
        
        samples_int16 = (samples * 32767).astype(np.int16)
        return AudioSegment(samples_int16.tobytes(), frame_rate=sr, sample_width=2, channels=1)
    
    except Exception as e:
        # Jika enhancement gagal, return original chunk
        print(f"      ⚠️  Enhancement error, using original: {str(e)}")
        return chunk_audio


# ========== MAIN: Extract Syllables dari Specific Run ==========
def extract_syllables_from_run(run_index: int = None, target_syllables: set = None):
    """
    Extract audio chunks untuk setiap syllable berdasarkan timing dari CSV.
    
    ⚠️ PENTING - Indexing SESUAI SCRAPER RUN:
    - INPUT: Membaca dari scraper run [run_index] 
      (syllable_dataset_[N].csv, temp_audio_[N]/)
    - OUTPUT: Menyimpan ke extracted_syllables_[run_index] ← SAMA DENGAN SCRAPER RUN INDEX
    - Jika folder sudah ada, akan OVERWRITE dengan data terbaru
    
    Timing Extraction:
    - Menggunakan syllable_start_sec dan syllable_end_sec dari CSV
    - Contoh: syllable 'ba' → potong dari 1.234s sampai 1.456s
    - Enhance dengan time stretch, denoise, normalize
    
    Args:
        run_index: Index dari scraper run (untuk membaca dan menulis)
                   Jika None, gunakan latest run
        target_syllables: Set of target syllables
                         Jika None, gunakan TARGET_SYLLABLES_EXTRACTION
    """
    if target_syllables is None:
        target_syllables = TARGET_SYLLABLES_EXTRACTION
    
    # Auto-detect latest run jika tidak specified
    if run_index is None:
        run_info = get_run_info(config.BASE_SCRAPED_DIR)
        if run_info['total_runs'] == 0:
            print("❌ Belum ada scraper runs!")
            return None
        run_index = max(run_info['runs'])
    
    print(f"\n{'='*70}")
    print(f"📁 Extracting Audio Syllables from Scraper Run #{run_index}")
    print(f"{'='*70}")
    
    # Setup INPUT paths (membaca dari scraper run)
    csv_file = os.path.join(config.BASE_SCRAPED_DIR, f"syllable_dataset_{run_index}.csv")
    temp_audio_dir = os.path.join(config.BASE_SCRAPED_DIR, f"temp_audio_{run_index}")
    
    # Setup OUTPUT dengan INDEX SAMA SEPERTI SCRAPER RUN
    # ⚠️ Jika folder sudah ada, akan OVERWRITE
    extracted_index = run_index
    output_dir = os.path.join(config.BASE_SCRAPED_DIR, f"extracted_syllables_{extracted_index}")
    
    print(f"\n📊 Paths:")
    print(f"   Input CSV: syllable_dataset_{run_index}.csv")
    print(f"   Input Audio Dir: temp_audio_{run_index}/")
    print(f"   Output Dir: extracted_syllables_{extracted_index}/ ← SESUAI RUN INDEX")
    
    if os.path.exists(output_dir):
        print(f"   ⚠️  Folder sudah ada → akan OVERWRITE dengan data terbaru")
    
    # ========== VALIDATE FILES ==========
    if not os.path.exists(csv_file):
        print(f"\n❌ CSV file tidak ditemukan: {csv_file}")
        return None
    
    if not os.path.exists(temp_audio_dir):
        print(f"\n❌ Temp audio dir tidak ditemukan: {temp_audio_dir}")
        return None
    
    # Find audio file
    audio_path = None
    for f in os.listdir(temp_audio_dir):
        if f.endswith("_cleaned.wav"):
            audio_path = os.path.join(temp_audio_dir, f)
            break
    
    if not audio_path:
        print(f"\n❌ Audio file tidak ditemukan di: {temp_audio_dir}")
        return None
    
    print(f"   Audio file: {os.path.basename(audio_path)}")
    
    # ========== LOAD DATA ==========
    print(f"\n⏳ Loading data...")
    df = pd.read_csv(csv_file)
    df_target = df[df['syllable'].isin(target_syllables)].copy()
    
    print(f"✓ Total rows dalam CSV: {len(df)}")
    print(f"✓ Target syllables found: {len(df_target)}")
    
    if df_target.empty:
        print(f"\n❌ Tidak ada target syllables di dataset!")
        return None
    
    # ========== LOAD AUDIO ==========
    print(f"\n⏳ Loading audio file...")
    try:
        audio = AudioSegment.from_wav(audio_path)
        audio_duration_sec = len(audio) / 1000.0
        print(f"✓ Audio duration: {audio_duration_sec:.2f}s")
    except Exception as e:
        print(f"\n❌ Error loading audio: {str(e)}")
        return None
    
    # Create output directories (atau clear jika sudah ada)
    if os.path.exists(output_dir):
        import shutil
        shutil.rmtree(output_dir)
        print(f"\n🧹 Cleared existing folder: {output_dir}")
    
    os.makedirs(output_dir, exist_ok=True)
    
    # ========== EXTRACT CHUNKS ==========
    print(f"\n⏳ Extracting syllable chunks based on timing from CSV...")
    extracted_count = 0
    skipped_count = 0
    label_counts = {}
    
    for idx, row in df_target.iterrows():
        try:
            syll = row['syllable']
            
            # PENTING: Gunakan syllable_start_sec dan syllable_end_sec dari CSV
            start_sec = float(row['syllable_start_sec'])
            end_sec = float(row['syllable_end_sec'])
            
            # Konversi ke milliseconds untuk pydub
            start_ms = int(start_sec * 1000)
            end_ms = int(end_sec * 1000)
            
            duration_sec = end_sec - start_sec
            
            # ========== VALIDASI TIMING ==========
            # Check jika timing valid
            if start_sec < 0:
                skipped_count += 1
                continue
            
            if end_sec > audio_duration_sec:
                skipped_count += 1
                continue
            
            if start_sec >= end_sec:
                skipped_count += 1
                continue
            
            if duration_sec < 0.05:  # Skip chunk yang terlalu pendek (< 50ms)
                skipped_count += 1
                continue
            
            # ========== EXTRACT CHUNK ==========
            # Slice audio menggunakan timing dari CSV
            chunk = audio[start_ms:end_ms]
            
            if len(chunk) == 0:
                skipped_count += 1
                continue
            
            # ========== ENHANCE CHUNK ==========
            chunk_enhanced = enhance_audio_chunk(chunk)
            
            # ========== SAVE FILE ==========
            # Buat folder untuk syllable label
            syll_dir = os.path.join(output_dir, syll)
            os.makedirs(syll_dir, exist_ok=True)
            
            # Buat nama file yang informatif
            # Format: [word]_[start_time]_[duration].wav
            word = str(row.get('original_word', f'word_{idx}'))
            filename = f"{word}_{start_sec:.3f}s_({duration_sec:.3f}s).wav"
            
            file_path = os.path.join(syll_dir, filename)
            chunk_enhanced.export(file_path, format="wav")
            
            extracted_count += 1
            label_counts[syll] = label_counts.get(syll, 0) + 1
            
        except Exception as e:
            # Skip jika ada error, lanjut ke chunk berikutnya
            skipped_count += 1
            continue
    
    # ========== SUMMARY REPORT ==========
    print(f"\n✅ Extraction Complete!")
    print(f"\n📊 Results:")
    print(f"   Total extracted: {extracted_count} files")
    print(f"   Skipped (invalid timing/short): {skipped_count}")
    print(f"   Labels extracted: {len(label_counts)}")
    print(f"   Output dir: {output_dir}")
    
    print(f"\n📈 Per-label breakdown:")
    for label in sorted(label_counts.keys()):
        count = label_counts[label]
        print(f"   - {label}: {count} files")
    
    print(f"\n💾 Directory structure:")
    print(f"   extracted_syllables_{extracted_index}/")
    for label in sorted(label_counts.keys()):
        print(f"   ├── {label}/ ({label_counts[label]} files)")
    
    return {
        "extracted_index": extracted_index,
        "run_index": run_index,
        "output_dir": output_dir,
        "total_extracted": extracted_count,
        "total_skipped": skipped_count,
        "label_counts": label_counts
    }

print("\n✓ Helper functions ready:")
print("   - enhance_audio_chunk()")
print("   - extract_syllables_from_run(run_index)")
print("\n💡 NOTE: extracted_syllables index sekarang SESUAI dengan scraper run index")
print("   Scraper run 1 → extracted_syllables_1")
print("   Scraper run 2 → extracted_syllables_2")
print("   (existing folders akan OVERWRITE)")

✓ Extraction config loaded
✓ Target syllables: 20 labels
✓ Target min duration: 0.3s

✓ Helper functions ready:
   - enhance_audio_chunk()
   - extract_syllables_from_run(run_index)

💡 NOTE: extracted_syllables index sekarang SESUAI dengan scraper run index
   Scraper run 1 → extracted_syllables_1
   Scraper run 2 → extracted_syllables_2
   (existing folders akan OVERWRITE)


### Extract Audio Syllables
Ekstraksi audio syllables dengan indexing yang sesuai dengan scraper run.

**Cara Menggunakan:**
- Kosongkan `run_index` atau set `None` → Auto-detect latest scraper run
- Atau set `run_index = N` → Extract dari scraper run #N spesifik

**Indexing:**
- `extracted_syllables_[N]` = hasil ekstraksi dari scraper run #[N]
- ⚠️ Jika folder sudah ada, akan OVERWRITE dengan data terbaru
- Contoh: run_index = 2 → extracted_syllables_2/ (jika ada, overwrite)

**Output:**
- ✅ Konfirmasi ekstraksi berhasil
- 📊 Jumlah file per label
- 📁 Lokasi output folder (sesuai run index)
- 📈 Breakdown per syllable label



In [9]:
# ========== Extract dari Specific Run ==========

# Pilihan run index
run_index_to_extract = None  # Kosongkan/None untuk auto-detect latest run
# Atau set ke nomor tertentu: run_index_to_extract = 1

print(f"\n{'='*70}")
print("🎵 OPTION 1: Extract Audio Syllables (Single Run)")
print(f"{'='*70}")

if run_index_to_extract is None:
    print(f"\n💡 run_index_to_extract = None → Auto-detect latest scraper run")
else:
    print(f"\n💡 run_index_to_extract = {run_index_to_extract}")

# Run extraction
result = extract_syllables_from_run(run_index=run_index_to_extract)

if result:
    print(f"\n{'='*70}")
    print("✅ EXTRACTION SUCCESSFUL!")
    print(f"{'='*70}")
    print(f"\n📊 Summary:")
    print(f"   Scraper Run: #{result['run_index']}")
    print(f"   Extracted Index: #{result['extracted_index']}")
    print(f"   Total Files Extracted: {result['total_extracted']}")
    print(f"   Total Skipped: {result['total_skipped']}")
    print(f"   Total Labels: {len(result['label_counts'])}")
    print(f"   Output: {result['output_dir']}")
    print(f"\n🎯 Extraction Details:")
    for label, count in sorted(result['label_counts'].items()):
        print(f"   - {label}: {count} audio chunks")
    print(f"\n✨ Ready untuk augmentation.ipynb! 🚀")
else:
    print(f"\n❌ Extraction gagal. Pastikan:")
    print(f"   1. Dataset sudah di-generate (run scraping pipeline terlebih dahulu)")
    print(f"   2. CSV file ada: syllable_dataset_[N].csv")
    print(f"   3. Temp audio folder ada: temp_audio_[N]/ dengan file *_cleaned.wav")
    print(f"   4. Timing data valid di CSV (syllable_start_sec < syllable_end_sec)")



🎵 OPTION 1: Extract Audio Syllables (Single Run)

💡 run_index_to_extract = None → Auto-detect latest scraper run

📁 Extracting Audio Syllables from Scraper Run #1

📊 Paths:
   Input CSV: syllable_dataset_1.csv
   Input Audio Dir: temp_audio_1/
   Output Dir: extracted_syllables_1/ ← SESUAI RUN INDEX
   ⚠️  Folder sudah ada → akan OVERWRITE dengan data terbaru
   Audio file: original_audio_cleaned.wav

⏳ Loading data...
✓ Total rows dalam CSV: 1149
✓ Target syllables found: 289

⏳ Loading audio file...
✓ Audio duration: 867.60s

🧹 Cleared existing folder: scraped\extracted_syllables_1

⏳ Extracting syllable chunks based on timing from CSV...


c:\Users\ASUS\Documents\CC x DBS 2026 (DS)\Capstone Project\Heartz\data-science2\scraping\venv\Lib\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1760
  warnings.warn(
c:\Users\ASUS\Documents\CC x DBS 2026 (DS)\Capstone Project\Heartz\data-science2\scraping\venv\Lib\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1376
  warnings.warn(
c:\Users\ASUS\Documents\CC x DBS 2026 (DS)\Capstone Project\Heartz\data-science2\scraping\venv\Lib\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1392
  warnings.warn(
c:\Users\ASUS\Documents\CC x DBS 2026 (DS)\Capstone Project\Heartz\data-science2\scraping\venv\Lib\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1600
  warnings.warn(
c:\Users\ASUS\Documents\CC x DBS 2026 (DS)\Capstone Project\Heartz\data-science2\scr


✅ Extraction Complete!

📊 Results:
   Total extracted: 272 files
   Skipped (invalid timing/short): 17
   Labels extracted: 17
   Output dir: scraped\extracted_syllables_1

📈 Per-label breakdown:
   - a: 70 files
   - ba: 17 files
   - be: 29 files
   - bi: 3 files
   - bo: 5 files
   - bu: 3 files
   - e: 13 files
   - i: 18 files
   - ma: 56 files
   - me: 3 files
   - mu: 3 files
   - o: 1 files
   - pa: 15 files
   - pe: 2 files
   - pi: 3 files
   - pu: 13 files
   - u: 18 files

💾 Directory structure:
   extracted_syllables_1/
   ├── a/ (70 files)
   ├── ba/ (17 files)
   ├── be/ (29 files)
   ├── bi/ (3 files)
   ├── bo/ (5 files)
   ├── bu/ (3 files)
   ├── e/ (13 files)
   ├── i/ (18 files)
   ├── ma/ (56 files)
   ├── me/ (3 files)
   ├── mu/ (3 files)
   ├── o/ (1 files)
   ├── pa/ (15 files)
   ├── pe/ (2 files)
   ├── pi/ (3 files)
   ├── pu/ (13 files)
   ├── u/ (18 files)

✅ EXTRACTION SUCCESSFUL!

📊 Summary:
   Scraper Run: #1
   Extracted Index: #1
   Total Files Extr